# Support Vector Machine (SVM)
## Import Library

In [1]:
#maftuhfaisal_202431163
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
#maftuhfaisal_202431163
dataset = pd.read_csv('big-five-european-league.csv')
dataset

,Country,League,Club,Player Names,Matches_Played,Substitution,Mins,Goals,xG,xG Per Avg Match,Shots,OnTarget,Shots Per Avg Match,On Target Per Avg Match,Year
0,Spain,La Liga,(BET),Juanmi Callejon,19,16,1849,11,6.62,0.34,48,20,2.47,1.03,2016
1,Spain,La Liga,(BAR),Antoine Griezmann,36,0,3129,16,11.86,0.36,88,41,2.67,1.24,2016
2,Spain,La Liga,(ATL),Luis Suarez,34,1,2940,28,23.21,0.75,120,57,3.88,1.84,2016
3,Spain,La Liga,(CAR),Ruben Castro,32,3,2842,13,14.06,0.47,117,42,3.91,1.40,2016
4,Spain,La Liga,(VAL),Kevin Gameiro,21,10,1745,13,10.65,0.58,50,23,2.72,1.25,2016
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655,Netherlands,Eredivisie,(UTR),Gyrano Kerk,24,0,2155,10,7.49,0.33,50,18,2.20,0.79,2020
656,Netherlands,Eredivisie,(AJA),Quincy Promes,18,2,1573,12,9.77,0.59,56,30,3.38,1.81,2020
657,Netherlands,Eredivisie,(PSV),Denzel Dumfries,25,0,2363,7,5.72,0.23,45,14,1.81,0.56,2020
658,Netherlands,Eredivisie,NaN,Cyriel Dessers,26,0,2461,15,14.51,0.56,84,43,3.24,1.66,2020


In [3]:
#maftuhfaisal_202431163
dataset.columns = dataset.columns.str.strip()
dataset.columns

Index(['Country', 'League', 'Club', 'Player Names', 'Matches_Played',
       'Substitution', 'Mins', 'Goals', 'xG', 'xG Per Avg Match', 'Shots',
       'OnTarget', 'Shots Per Avg Match', 'On Target Per Avg Match', 'Year'],
      dtype='str')

In [4]:
#maftuhfaisal_202431163
dataset['Kategori'] = pd.cut(dataset['Goals'], bins = [0, 8, 13, 100],
                             labels = ['Rendah', 'Sedang', 'Tinggi']).astype(str)
dataset['Kategori'].unique()

<StringArray>
['Sedang', 'Tinggi', 'Rendah']
Length: 3, dtype: str

In [5]:
#maftuhfaisal_202431163
dataset['Kategori'].value_counts()

Kategori
Sedang    283
Tinggi    197
Rendah    180
Name: count, dtype: int64

In [6]:
#maftuhfaisal_202431163
le = LabelEncoder()
dataset['Kategori_encoded'] = le.fit_transform(dataset['Kategori'])

fitur = ['Matches_Played', 'Substitution', 'Mins', 'xG', 'Shots', 'OnTarget']
x = dataset[fitur]
y = dataset['Kategori_encoded']

In [7]:
#maftuhfaisal_202431163
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3, random_state=0, stratify=y)

In [8]:
#maftuhfaisal_202431163
pipeline = Pipeline([('scaler', StandardScaler()),('svc',SVC())])

In [9]:
#maftuhfaisal_202431163
params = {
    'svc__kernel': ['poly', 'rbf', 'sigmoid'],
    'svc__C': [0.5, 1, 10, 100],
    'svc__gamma': ['scale', 1, 0.1, 0.001]
}

In [10]:
#maftuhfaisal_202431163
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=params,
    scoring='accuracy',
    cv=5,
    n_jobs=1,
    verbose=1
)

In [11]:
#maftuhfaisal_202431163
grid_search.fit(x_train, y_train)

best_model = grid_search.best_estimator_

y_train_pred = best_model.predict(x_train)
y_test_pred = best_model.predict(x_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Best Parameters:", grid_search.best_params_)
print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best Parameters: {'svc__C': 100, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'}
Train Accuracy: 0.816017316017316
Test Accuracy: 0.7474747474747475


In [12]:
#maftuhfaisal_202431163
print("\nClassification Report (Test):\n",
    classification_report(y_test, y_test_pred, target_names=le.classes_))

print(
    "\nConfusion Matrix (Test):\n", confusion_matrix(y_test, y_test_pred))


Classification Report (Test):
               precision    recall  f1-score   support

      Rendah       0.86      0.80      0.83        54
      Sedang       0.68      0.79      0.73        85
      Tinggi       0.78      0.64      0.70        59

    accuracy                           0.75       198
   macro avg       0.77      0.74      0.75       198
weighted avg       0.76      0.75      0.75       198


Confusion Matrix (Test):
 [[43 11  0]
 [ 7 67 11]
 [ 0 21 38]]


In [13]:
#maftuhfaisal_202431163
pemain_baru = pd.DataFrame([[30, 3, 2500, 14, 90, 40]], columns = fitur)
pred = best_model.predict(pemain_baru)
print("Predicted label (encoded): ", pred)
print("Predicted Kategori : ", le.inverse_transform(pred))

Predicted label (encoded):  [2]
Predicted Kategori :  ['Tinggi']
